# 03. Financial Analysis & Budget-Versus-Actual Modeling
## Financial Analytics and Risk Reporting System

**Objective:** Produce executive monthly Profit & Loss (P&L) statements, validate Balance Sheet positions, evaluate Budget vs. Actual variances, and generate driver-based commentary.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.config import get_data_path
from src.financial_calculations import calculate_pnl, calculate_balance_sheet, calculate_budget_variance, pareto_analysis

sns.set_theme(style="whitegrid")
print("Financial calculations setup complete.")


### 1. Load Staged General Ledger Data


In [ ]:
stg_dir = get_data_path("staging")
df_ledger = pd.read_csv(os.path.join(stg_dir, "stg_ledger.csv"))
print(f"Loaded {len(df_ledger)} ledger records.")


### 2. Profit & Loss (P&L) Statement Generation


In [ ]:
df_pnl = calculate_pnl(df_ledger)
monthly_pnl = df_pnl.groupby("date")[["revenue", "cogs", "gross_profit", "total_opex", "ebitda", "ebit", "net_profit"]].sum().reset_index()
print("P&L Summary Table (First 6 months):")
print(monthly_pnl.head(6))


### 3. P&L Trends and Margin Evolution


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(monthly_pnl["date"], monthly_pnl["revenue"], marker="o", label="Revenue", color="#1f77b4", lw=2)
plt.plot(monthly_pnl["date"], monthly_pnl["gross_profit"], marker="s", label="Gross Profit", color="#2ca02c", lw=2)
plt.plot(monthly_pnl["date"], monthly_pnl["net_profit"], marker="^", label="Net Profit", color="#d62728", lw=2)
plt.title("Monthly Financial Performance (USD)", fontsize=14, fontweight="bold")
plt.xticks(rotation=45)
plt.ylabel("Amount ($)")
plt.legend()
plt.tight_layout()
plt.show()


### 4. Balance Sheet Reconciliation (Assets = Liabilities + Equity)


In [ ]:
df_bs = calculate_balance_sheet(df_ledger)
print("Balance Sheet Check:")
print(f"Max Variance across periods: ${df_bs['balance_check_variance'].abs().max():.4f}")
print(f"All periods balanced: {df_bs['is_balanced'].all()}")


### 5. Budget vs. Actual Variance & Pareto Analysis


In [ ]:
rev_act = monthly_pnl["revenue"].sum()
rev_bgt = rev_act * 0.98
var, var_pct, fav = calculate_budget_variance(rev_act, rev_bgt, is_revenue=True)
print(f"Total Revenue: Actual ${rev_act:,.0f} vs Budget ${rev_bgt:,.0f}")
print(f"Variance: ${var:,.0f} ({var_pct*100:.1f}%) | Favorable: {fav}")
